# 06. 1D Convolutional Neural Network (1D-CNN) Model
**ML-Powered Intrusion Detection System (IDS) for Secure Network Monitoring**

This notebook implements, trains, and evaluates a 1D-CNN for network intrusion detection. The 1D-CNN processes ordered network flow feature vectors as 1D sequences to extract local hierarchical patterns.

## 1. Imports & Environment Setup

In [ ]:
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Ensure project root is in sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.models.cnn_1d import Conv1DModel
from src.preprocessing.verify_split import load_partition

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "models" / "cnn_1d"
METRICS_DIR = ROOT_DIR / "results" / "metrics"
GRAPHS_DIR = ROOT_DIR / "results" / "graphs"
CONF_DIR = ROOT_DIR / "results" / "confusion_matrices"

## 2. Dataset Loading

In [ ]:
try:
    X_train, y_train = load_partition(PROCESSED_DIR / "train", "train")
    X_val, y_val = load_partition(PROCESSED_DIR / "validation", "val")
    X_test, y_test = load_partition(PROCESSED_DIR / "test", "test")
    
    mapping_file = PROCESSED_DIR / "label_mapping.json"
    label_mapping = {}
    if mapping_file.exists():
        with open(mapping_file, "r", encoding="utf-8") as f:
            label_mapping = json.load(f)
    inv_mapping = {v: k for k, v in label_mapping.items()}
    target_names = [inv_mapping.get(i, f"Class_{i}") for i in sorted(np.unique(y_train))]
    
    print(f"Loaded Partitions Successfully:")
    print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
    print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")
    print(f"  Number of Classes: {len(target_names)}")
except FileNotFoundError:
    print("Processed dataset partitions not found. Execute preprocessing pipeline on raw data in data/raw/.")

## 3. Input Representation & Reshaping
The 1D-CNN processes the ordered tabular feature vector as a 1D sequence.
We reshape 2D feature matrices `(samples, features)` into 3D tensors `(samples, features, 1)` where the 3rd dimension represents the single input channel.

In [ ]:
if 'X_train' in locals():
    num_features = X_train.shape[1]
    num_classes = len(target_names)
    
    X_train_cnn = np.expand_dims(X_train, axis=-1) if X_train.ndim == 2 else X_train
    X_val_cnn = np.expand_dims(X_val, axis=-1) if X_val.ndim == 2 else X_val
    X_test_cnn = np.expand_dims(X_test, axis=-1) if X_test.ndim == 2 else X_test
    
    print(f"Reshaped Tensor Dimensions for Conv1D:")
    print(f"  X_train_cnn: {X_train_cnn.shape}")
    print(f"  X_val_cnn:   {X_val_cnn.shape}")
    print(f"  X_test_cnn:  {X_test_cnn.shape}")

## 4. Class Distribution Analysis

In [ ]:
if 'y_train' in locals():
    train_counts = pd.Series(y_train).value_counts().sort_index()
    train_dist = pd.DataFrame({
        "Class_Index": train_counts.index,
        "Class_Name": [inv_mapping.get(i, f"Class_{i}") for i in train_counts.index],
        "Train_Samples": train_counts.values,
        "Percentage": (train_counts.values / len(y_train)) * 100
    })
    display(train_dist)

## 5. 1D-CNN Architecture Construction & Model Summary

In [ ]:
if 'num_features' in locals():
    cnn_model = Conv1DModel(
        num_features=num_features,
        num_classes=num_classes,
        filters=(64, 128),
        kernel_size=3,
        dense_units=128,
        dropout_rate=0.3,
        learning_rate=0.001,
    )
    keras_cnn = cnn_model.build_model()
    keras_cnn.summary()

## 6. Model Training & Callbacks

In [ ]:
if 'keras_cnn' in locals() and 'X_train' in locals():
    import tensorflow as tf
    
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = MODELS_DIR / "best_model.keras"
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=str(checkpoint_path), monitor="val_loss", save_best_only=True, verbose=1),
    ]
    
    print("Training 1D-CNN on X_train, y_train and evaluating on X_val, y_val...")
    t0 = time.time()
    history = cnn_model.fit(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        epochs=30,
        batch_size=128,
        callbacks_list=callbacks,
    )
    t_train = time.time() - t0
    print(f"1D-CNN Training completed in {t_train:.2f} seconds.")

## 7. Training & Validation Curves

In [ ]:
if 'history' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss Curve
    axes[0].plot(history.history['loss'], label='Train Loss', color='#2b5c8f', lw=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e27c3e', lw=2)
    axes[0].set_title('1D-CNN — Training vs. Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy Curve
    axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='#2b5c8f', lw=2)
    axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='#3ca35d', lw=2)
    axes[1].set_title('1D-CNN — Training vs. Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Validation Set Evaluation

In [ ]:
if 'cnn_model' in locals() and cnn_model.model is not None:
    y_val_pred = cnn_model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1_w = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
    val_f1_m = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    
    print(f"Validation Set Metrics:")
    print(f"  Accuracy:    {val_acc*100:.2f}%")
    print(f"  Weighted F1: {val_f1_w:.4f}")
    print(f"  Macro F1:    {val_f1_m:.4f}")

## 9. Final Test Set Evaluation

In [ ]:
if 'cnn_model' in locals() and cnn_model.model is not None:
    y_test_pred = cnn_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1_w = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
    test_f1_m = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
    
    print(f"Final Test Set Metrics (Unbiased Single Evaluation):")
    print(f"  Accuracy:    {test_acc*100:.2f}%")
    print(f"  Weighted F1: {test_f1_w:.4f}")
    print(f"  Macro F1:    {test_f1_m:.4f}")

## 10. Confusion Matrix

In [ ]:
if 'y_test_pred' in locals():
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=target_names, yticklabels=target_names)
    plt.title('1D-CNN — Test Set Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 11. Per-Class Performance Metrics

In [ ]:
if 'y_test_pred' in locals():
    report_dict = classification_report(y_test, y_test_pred, target_names=target_names, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    display(report_df)

## 12. Misclassification & Error Analysis

In [ ]:
if 'y_test_pred' in locals():
    err_idx = np.where(y_test != y_test_pred)[0]
    print(f"Total Misclassified Samples: {len(err_idx)} ({len(err_idx)/len(y_test)*100:.2f}% error rate)")
    
    err_df = pd.DataFrame({
        "Sample_Idx": err_idx,
        "True_Class": [inv_mapping.get(y_test[i], str(y_test[i])) for i in err_idx],
        "Predicted_Class": [inv_mapping.get(y_test_pred[i], str(y_test_pred[i])) for i in err_idx],
    })
    display(err_df.head(15))

## 13. Comparison with Random Forest and XGBoost Baselines

In [ ]:
comp_file = METRICS_DIR / "model_comparison.csv"
if comp_file.exists():
    comp_df = pd.read_csv(comp_file)
    display(comp_df)
else:
    print("Model comparison table not yet available. Run scripts/train_cnn_1d.py to populate benchmarks.")